In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
functional_zones = gpd.read_file("../data/blocksnet/functional_zones.geojson")
functional_zones

In [ ]:
cadastr_blocks = gpd.read_parquet('../data/traning_data/spb_bloks_price_train.parquet')
cadastr_blocks = cadastr_blocks.to_crs(epsg=32636)
cadastr_blocks

In [ ]:
import pandas as pd
import geopandas as gpd


def postprocess_urban_blocks_keep_attrs(blocks: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if not isinstance(blocks, gpd.GeoDataFrame):
        raise ValueError("An instance of GeoDataFrame must be provided")
    if len(blocks) == 0:
        raise ValueError("Rows count must be greater than 0")

    blocks = blocks.copy()

    valid_blocks = blocks[blocks.is_valid].copy()
    invalid_blocks = blocks[~blocks.is_valid].copy()

    if not invalid_blocks.empty:
        invalid_blocks["geometry"] = invalid_blocks.geometry.make_valid()

    blocks = gpd.GeoDataFrame(
        pd.concat([valid_blocks, invalid_blocks], ignore_index=True),
        crs=blocks.crs,
    )

    blocks = blocks.explode(index_parts=False).reset_index(drop=True)
    blocks = blocks[blocks.geom_type == "Polygon"].copy()
    blocks = blocks[~blocks.geometry.is_empty].copy()
    blocks = blocks.reset_index(drop=True)

    return blocks


cadastr_blocks = postprocess_urban_blocks_keep_attrs(cadastr_blocks)
cadastr_blocks

In [ ]:
# from blocksnet.blocks.postprocessing import postprocess_urban_blocks
# cadastr_geometry = postprocess_urban_blocks(cadastr_blocks)
# cadastr_geometry

In [ ]:
cadastr_blocks['log_cost_index'] = np.log1p(cadastr_blocks['cost_index'])

In [ ]:
# Картируем лог-цену
cadastr_blocks.plot(
    column='log_cost_index',
    scheme='quantiles',
    legend=True,
    figsize=(15,15),
    cmap='viridis'
).set_axis_off()
plt.title('Карта кадастровой стоимости земельных участков в Санкт-Петербурге (логарифм от стоимости м2)')

plt.show()

# Назначение функциональных зон

In [ ]:
functional_zones = functional_zones.to_crs(epsg=32636)

In [ ]:

from blocksnet.blocks.assignment import assign_objects

objects_gdf = assign_objects(cadastr_blocks, functional_zones.rename(columns={'functional_zone': 'name'}))
objects_gdf.head()

In [ ]:
objects_gdf.plot(
    column='name',
    figsize=(15,15),
    legend=True,
    cmap='tab20'
).set_axis_off()
plt.title('Карта функциональных зон в Санкт-Петербурге')
plt.show()

In [ ]:
from blocksnet.enums import LandUse

rules = {
    "Т3Ж1": LandUse.RESIDENTIAL,
    "ТР0-2": LandUse.RECREATION,
    "Т3Ж2": LandUse.RESIDENTIAL,
    "Т1Ж2-1": LandUse.RESIDENTIAL,
    "Т2ЖД2": LandUse.RESIDENTIAL,
    "ТД1-3": LandUse.BUSINESS,
    "ТД2": LandUse.BUSINESS,
    "ТД3": LandUse.BUSINESS,
    "ТУ": LandUse.TRANSPORT,
    "ТИ4": LandUse.TRANSPORT,
    "ТД1-1": LandUse.RESIDENTIAL,
    "ТД1-2": LandUse.RESIDENTIAL,
    "ТПД1": LandUse.INDUSTRIAL,
    "ТПД2": LandUse.INDUSTRIAL,
    "ТИ1-1": LandUse.TRANSPORT,
    "Т3ЖД3": LandUse.RESIDENTIAL,
    "ТК1": LandUse.SPECIAL,
    "ТР2": LandUse.RECREATION,
    "ТИ2": LandUse.TRANSPORT,
    "ТР5-2": LandUse.RECREATION,
    "Т1Ж2-2": LandUse.RESIDENTIAL,
    "ТР4": LandUse.RECREATION,
    "ТР5-1": LandUse.RECREATION,
    "Т2Ж1": LandUse.RESIDENTIAL,
    "ТИ3": LandUse.TRANSPORT,
    "Т1Ж1": LandUse.RESIDENTIAL,
    "ТИ1-2": LandUse.TRANSPORT,
    "ТР3-2": LandUse.RECREATION,
    "ТР0-1": LandUse.RECREATION,
    "ТП2": LandUse.INDUSTRIAL,
    "ТК3": LandUse.SPECIAL,
    "ТР1": LandUse.RECREATION,
    "ТР3-1": LandUse.RECREATION,
    "ТС1": LandUse.AGRICULTURE,
    "ТК2": LandUse.SPECIAL,
    "ТП1": LandUse.INDUSTRIAL,
    "ТП3": LandUse.INDUSTRIAL,
    "ТП4": LandUse.INDUSTRIAL,
    "ТС2": LandUse.SPECIAL,
}

In [ ]:
import math

math.ceil(1/1000*16)

In [ ]:
from blocksnet.blocks.assignment import assign_land_use

land_use_gdf = assign_land_use(cadastr_blocks, functional_zones, rules)
land_use_gdf.head()

In [ ]:
land_use_gdf.plot(column='land_use', legend=True, figsize=(10,10)).set_axis_off()

In [ ]:
cadastr_blocks = cadastr_blocks.join(land_use_gdf.drop(columns=['geometry']))
cadastr_blocks.head()


# Назначение зданий

In [ ]:
buildings = gpd.read_file("../data/blocksnet/buildings.geojson")
buildings = buildings.to_crs(epsg=32636)
buildings['number_of_floors'] = buildings['storeys_count'].clip(lower=1)
buildings['population'] = buildings['population_balanced']
buildings

In [ ]:
from blocksnet.preprocessing.imputing import impute_buildings
buildings = impute_buildings(buildings, default_living_demand=30)
buildings['number_of_floors'] = buildings['number_of_floors'].clip(lower=1)
buildings

In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects

buildings_blocks = aggregate_objects(cadastr_blocks, buildings)[0]

In [ ]:
buildings_blocks

In [ ]:
cadastr_blocks = cadastr_blocks.join(buildings_blocks.drop(columns=['geometry']))

# Морфотипы 

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

cadastr_blocks['site_area'] = cadastr_blocks['specified_area']
density_input = cadastr_blocks[
    ['site_area', 'footprint_area', 'build_floor_area', 'living_area']
].copy()

density_input = density_input.dropna(subset=['site_area', 'footprint_area', 'build_floor_area', 'living_area'])

density_input['non_living_area'] = (
    density_input['build_floor_area'] - density_input['living_area']
).clip(lower=0)

density_df = calculate_density_indicators(density_input)
density_df


In [ ]:
cadastr_blocks.loc[:, density_df.columns] = density_df
cadastr_blocks.head()

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes


blocks_df = get_strelka_morphotypes(cadastr_blocks)
blocks_df.head()

In [ ]:
cadastr_blocks.loc[:, blocks_df.columns] = blocks_df
cadastr_blocks.head()

# Связанность

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

from urbanomy.methods.land_value_modeling.constants import ACCESSIBILITY_SPEED


def calculate_area_accessibility_grid_approx(
    blocks: gpd.GeoDataFrame,
    site_area_col: str = "site_area",
    grid_size: float = 500.0,
    chunk_size: int = 2048,
) -> pd.DataFrame:
    if not isinstance(blocks, gpd.GeoDataFrame):
        raise ValueError("blocks must be a GeoDataFrame")
    if blocks.empty:
        return pd.DataFrame(index=blocks.index, columns=["area_accessibility"], dtype=float)
    if site_area_col not in blocks.columns:
        raise ValueError(f"Column '{site_area_col}' not found")
    if blocks.crs is None:
        raise ValueError("GeoDataFrame CRS is None")

    utm_crs = blocks.estimate_utm_crs()
    work = blocks.to_crs(utm_crs) if utm_crs else blocks.copy()

    reps = work.geometry.representative_point()
    x = reps.x.to_numpy(dtype=np.float64)
    y = reps.y.to_numpy(dtype=np.float64)

    site_area = work[site_area_col].astype(float).fillna(0).to_numpy(dtype=np.float64)
    total_area = site_area.sum()
    if total_area <= 0:
        raise ValueError("Sum of site_area must be positive")

    grid_x = np.floor(x / grid_size).astype(np.int64)
    grid_y = np.floor(y / grid_size).astype(np.int64)

    df = pd.DataFrame(
        {
            "grid_x": grid_x,
            "grid_y": grid_y,
            "x": x,
            "y": y,
            "site_area": site_area,
        },
        index=blocks.index,
    )

    cells = (
        df.groupby(["grid_x", "grid_y"], as_index=False)
        .agg(
            site_area=("site_area", "sum"),
            x=("x", "mean"),
            y=("y", "mean"),
        )
    )

    cell_x = cells["x"].to_numpy(dtype=np.float64)
    cell_y = cells["y"].to_numpy(dtype=np.float64)
    cell_area = cells["site_area"].to_numpy(dtype=np.float64)

    result = np.empty(len(df), dtype=np.float64)

    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))

        dx = x[start:end, None] - cell_x[None, :]
        dy = y[start:end, None] - cell_y[None, :]
        dist = np.sqrt(dx * dx + dy * dy)

        time_matrix = np.floor(dist / ACCESSIBILITY_SPEED)
        weighted_sum = (time_matrix * cell_area[None, :]).sum(axis=1)
        result[start:end] = weighted_sum / total_area

    return pd.DataFrame({"area_accessibility": result}, index=blocks.index)



In [ ]:
if "site_area" not in cadastr_blocks.columns:
    projected = cadastr_blocks.to_crs(cadastr_blocks.estimate_utm_crs())
    cadastr_blocks["site_area"] = projected.geometry.area

area_acc_df = calculate_area_accessibility_grid_approx(
    cadastr_blocks,
    site_area_col="site_area",
    grid_size=250,
    chunk_size=2048,
)



In [ ]:
cadastr_blocks["area_accessibility"] = area_acc_df["area_accessibility"]
cadastr_blocks["land_value"] = cadastr_blocks["cost_value"]
cadastr_blocks["land_value_per_sqm"] = cadastr_blocks["cost_index"]
cadastr_blocks["log_land_value"] = np.log1p(cadastr_blocks["land_value"])
cadastr_blocks["log_land_value_per_sqm"] = np.log1p(cadastr_blocks["land_value_per_sqm"])
cadastr_blocks.head()

In [ ]:
# Очистка базовых невалидных значений
cadastr_blocks = cadastr_blocks.replace([np.inf, -np.inf], np.nan).copy()

# Ключевые поля в число
for col in ["cost_value", "site_area"]:
    cadastr_blocks[col] = pd.to_numeric(cadastr_blocks[col], errors="coerce")

# Удаляем все кварталы, где cost_value <= 0 или NaN
cadastr_blocks= cadastr_blocks[cadastr_blocks["cost_value"] > 1].copy()

# (Рекомендовано) также убрать нулевую/битую площадь, чтобы деление было корректным
cadastr_blocks = cadastr_blocks[cadastr_blocks["site_area"] > 1].copy()


# Быстрая статистика
print(cadastr_blocks['cost_value'].describe())
    

In [ ]:
blocks_pred = cadastr_blocks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# 2) Заменяем inf на NaN
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan)
blocks_pred = blocks_pred.fillna(0)

# 3) Удаляем выбросы: оставляем данные до 99-го перцентиля
p99 = blocks_pred["land_value_per_sqm"].quantile(0.99)
blocks_clean = blocks_pred[blocks_pred["land_value_per_sqm"] <= p99].copy()

print(blocks_clean["land_value_per_sqm"].describe())

# 4) Гистограмма после очистки
plt.figure()
blocks_clean["land_value_per_sqm"].dropna().hist(bins=50)
plt.xlabel("Цена за сотку (руб.)")
plt.ylabel("Частота")
plt.title("Распределение цены за сотку")
plt.show()

# 5) Box-plot после очистки
plt.figure()
plt.boxplot(blocks_clean["land_value_per_sqm"].dropna(), vert=False)
plt.xlabel("Цена за сотку (руб.)")
plt.title("Box-plot цены за сотку")
plt.show()


In [ ]:

# удалить несколько колонок
blocks_clean = blocks_clean.drop(columns=["cadastralDistrictsCode", "readable_address", "cost_value", "cost_index",
"land_record_category_type", "permitted_use_established_by_document","specified_area","log_cost_index", 'count'])


In [ ]:
blocks_clean.columns.to_list()

In [ ]:
# blocks_clean.to_file("../data/blocks_clean.geojson", driver="GeoJSON")


In [ ]:
import pandas as pd

# 1. Отбираем числовые столбцы и фильтруем
num = blocks_clean.select_dtypes(include=['number'])
num = num[num['land_value'] > 1]

# 2. Считаем count + статистики
agg_stats = num.agg(['count', 'min', 'max', 'mean', 'median', 'std']).T

# 3. Переименовываем столбцы
agg_stats.index.name = 'Variable'
agg_stats = agg_stats.rename(columns={
    'count':  'Count',
    'min':    'Min',
    'max':    'Max',
    'mean':   'Mean',
    'median': 'Median',
    'std':    'SD'
})

# 4. Приводим Count к int
agg_stats['Count'] = agg_stats['Count'].astype(int)

# 5. Округляем остальные метрики
for col in ['Min','Max','Mean','Median','SD']:
    agg_stats[col] = agg_stats[col].round(2)

# 6. Настраиваем глобальный формат для float
pd.options.display.float_format = '{:,.2f}'.format

# 7. Показываем результат
agg_stats


In [ ]:
import libpysal
from esda.moran import Moran, Moran_Local
import matplotlib.pyplot as plt
from splot.esda import lisa_cluster

# 1) Подготовка: spatial weights
# Если у вас уже есть w1 (Rook или Queen), используем его, иначе:
w1 = libpysal.weights.Queen.from_dataframe(blocks_default)
# или: w1 = libpysal.weights.Rook.from_dataframe(blocks)
w1.transform = 'r'   # row-standardization

# 2) Извлекаем вектор переменной
y = blocks_default['log_total_price'].values

# 3) Глобальный индекс Морена
moran_global = Moran(y, w1)
print("Global Moran’s I:",     round(moran_global.I,3))
print("p-value (normal):",     round(moran_global.p_norm,3))
print("z-score (normal):",     round(moran_global.z_norm,3))
print("p-value (permut.):",    round(moran_global.p_sim,3))
print("z-score (permut.):",    round(moran_global.z_sim,3))

# 4) Локальный индекс Морена
lisa = Moran_Local(y, w1)

# 5) Добавляем результаты в GeoDataFrame
blocks_default['lisa_I']   = lisa.Is          # локальные значения I
blocks_default['lisa_p']   = lisa.p_sim       # p-values по перестановкам
blocks_default['lisa_q']   = lisa.q           # квадранты (1=HH, 2=LH, 3=LL, 4=HL)
blocks_default['lisa_sig'] = lisa.p_sim < 0.05

# 6) Визуализация LISA-кластеров
fig, ax = plt.subplots(1, figsize=(15, 15))
# splot умеет сам раскрасить по четырём кластерам
lisa_cluster(lisa, blocks_default, p=0.05, ax=ax)
ax.set_title("LISA-кластеризация (p<0.05)")
ax.axis('off')
plt.show()
